<a href="https://colab.research.google.com/github/comparativechrono/computational_biology_notebooks/blob/main/Bioinformatics/Algorithms_11_networkx_and_protein_interactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Networkx and protein:protein interaction data

## Ribosomal protein interactions in Networkx

This notebook will be using data from a paper on protein networks in the eukaryotic ribosome. The original structure for this is pdb code 4v88 which has both the large (L) and small (S) subunits. The ribosome is basically an RNA machine for making proteins using the amino-acids provided by tRNAs. Protein decorate and stabilize the outside of the RNA - they also make contact with each other - very often through elongated projections that have been likened to the communicating extensions of neurones.

The data in this notebook are from the analysis of these extensions in the 4v88 data by Poirot and Timsit (2016, Neuron-Like Networks Between Ribosomal Proteins Within the Ribosome. Sci Rep. 2016;6:26485. https://doi.org/10.1038/srep26485)

Although it is interesting to look at the protein:protein interactions that they study in detail - it is important to remember that the ribosome is mainly RNA. You can see the structure if you load the 4v88 into ICM browser or at the PDBe webpage https://pdbe.org/4v88.

If you do you will see how the proteins decorate the outer surface of the subunits. And that there is a cleft between the two subunits that allows the mRNA access to the active site. Because of this cleft, relatively few proteins from the small subunit can reach those in the large subunit.

The tables of interactions have been turned into the python data structures below.

Here is the network of contacts mentioned in the paper above (extracted from their tables S3 and S4). Many proteins have several contacts through elongated extensions.

In [ ]:
# run this cell to load the table_of_contacts dictionary
table_of_contacts = {"eL13" : ["eL15", "eL18", "eL36", "uL15", "uL29", "uL4"],
"eL14" : ["eL20", "eL6", "uL13", "uL6"],
"eL15" : ["eL36", "eL42", "eL8", "uL2", "uL29", "uL4"],
"eL18" : ["uL15", "uL30", "uL4"],
"eL19" : ["eS7"],
"eL20" : ["eL21", "uL13", "uL16", "uL30", "uL4", "uL6"],
"eL21" : ["eL29", "uL16", "uL18", "uL30", "uL4"],
"eL24" : ["eS6", "uL14", "uL3"],
"eL27" : ["eL30", "eL34", "eL8"],
"eL30" : ["eL34", "eL43"],
"eL32" : ["eL33", "eL6", "uL15"],
"eL33" : ["eL6", "uL13", "uL22"],
"eL34" : ["eL39"],
"eL36" : ["eL8", "uL15"],
"eL37" : ["eL39", "uL29", "uL4"],
"eL39" : ["uL23", "uL24"],
"eL40" : ["uL6"],
"eL42" : ["uL15", "uL5"],
"eL43" : ["uL2"],
"eL8" : ["uL2", "uL23"],
"eS10" : ["eS12", "uS14", "uS3"],
"eS12" : ["eS31"],
"eS17" : ["uS2", "uS3"],
"eS19" : ["uS13", "uS9"],
"eS21" : ["eS24", "eS27", "eS4", "eS6", "uS2", "uS4", "uS5", "uS8"],
"eS24" : ["eS4", "eS6", "uS4"],
"eS25" : ["uS13", "uS7", "uS9"],
"eS27" : ["uS15", "uS8"],
"eS28" : ["eS30", "uS12", "uS4", "uS7"],
"eS30" : ["uS12", "uS4"],
"eS4" : ["eS6", "uS17", "uS4"],
"eS6" : ["uL3"],
"eS7" : ["uS15", "uS8"],
"eS8" : ["uS17"],
"uL13" : ["uL3", "uL6"],
"uL14" : ["uL3"],
"uL16" : ["uL18"],
"uL18" : ["uL5"],
"uL23" : ["uL29"],
"uL24" : ["uL4"],
"uL30" : ["uL4"],
"uS10" : ["uS14", "uS3", "uS9"],
"uS12" : ["uS17", "uS8"],
"uS13" : ["uS19"],
"uS14" : ["uS3"],
"uS15" : ["uS17", "uS8"],
"uS17" : ["uS8"],
"uS2" : ["uS5"],
"uS3" : ["uS5"],
"uS4" : ["uS5", "uS8"],
"uS5" : ["uS8"],
"uS7" : ["uS9"],
"GBP" : ["uS9", "uS12", "eS17"],
"STM1" : ["uS12", "uS19", "eS31" ],}

In [ ]:
# run this cell to import networkx and pyplot
import networkx as nx
import matplotlib.pyplot as plt

The protein interactions will be treated as reciprocal and so an undirected graph is used.

In [ ]:
# run this cell to create (an empty) nx.Graph called contacts_graph
contacts_graph = nx.Graph()

The individual proteins can be added as nodes from the `table_of_contacts` dictionary keys

In [ ]:
# run this cell add the individual proteins as nodes of protein_graph
contacts_graph.add_nodes_from(table_of_contacts.keys())

Running `nx.info` shows that after this the graph is a series of nodes that are not yet connected

In [ ]:
# run this cell to get basic info on contacts_graph
print(nx.info(contacts_graph))

Then the contacts from the network are added as edges.

In [ ]:
# Then we determine the edges on the nodes
for protein, interacting_proteins in table_of_contacts.items():
    for contact in interacting_proteins:
        contacts_graph.add_edge(protein, contact)

After we have add the edges it can be noted that there are edges and that the number of nodes in the graph has increased (question: why is this so?).

In [ ]:
# run this cell to get basic info on contacts_graph
print(nx.info(contacts_graph))

Lets try drawing the graph in two ways:

In [ ]:
nx.draw_random(contacts_graph,node_color='r',)

In [ ]:
nx.draw_circular(contacts_graph,node_color='y')

Neither one of those representations showed the effect of the cleft between the small and the large subunit.

One approach is to apply a force to the nodes. The nodes are treated as if each edge was a little spring. This can untangle them.

This has to be done iteratively. And like the random approach it can vary from run to run.

But it should separate out the clusters of nodes for the two subunits.

If not then try altering the constant k or the number of interations.

In [ ]:
nx.draw_spring(contacts_graph, k=0.80, iterations=30, with_labels=True, node_color='c',)

Note that each time the cell is run a different diagram is produced.

You should be able to see a separation between the small and large subunits. But it would be much clearer if the small and large subunits had distinct colors. The following cell has incomplete code to acheive this.

In [ ]:
# TASK!
# complete code to draw diagram with small and large nodes different colors
plt.figure(figsize=(10,10))
pos_spring = nx.spring_layout(contacts_graph)
small_node_list = [label  for label in set(contacts_graph) if 'S' in label]
nx.draw_networkx_nodes(contacts_graph, pos=pos_spring, nodelist=small_node_list, node_color='pink')
nx.draw_networkx_edges(contacts_graph, pos=pos_spring, alpha=0.5)
# complete code drawing the other 'large' nodes and labelling all nodes
plt.axis('off')

Question: there is one subunit called `GBP` (short for Guanine binding protein) that is does not have `S` or `L` in its name. In our first attempt at a diagram we grouped this with the large subunits. Draw a second diagram including GBP with the small subunits. Does it make more sense to included GBP with the small or with the large subunits?

In [ ]:
# your turn draw a network diagram colouring the small and large
# subunits in different colours including GBP as a small subunit

## Degree of the network

This is a useful metrix to understand how connected a network is.

In [ ]:
n = contacts_graph.number_of_nodes()
print(n)

In [ ]:
e = contacts_graph.number_of_edges()
print(e)

We can calculate the average degree by multipling twice the number of edges by the number of nodes. Why twice?Remember that each edge connects two nodes so will contribute to the degree of both.

In [ ]:
print('Average degree',  2*e/n)

The average degree of a network is included in the short summary of information of a graph provided by the function `nx.info`:

In [ ]:
print(nx.info(contacts_graph))

The network density compares this number of edges with the maximum possible.

In [ ]:
nx.density(contacts_graph)

In [ ]:
# run this cell to print out the degree for each node in contact_graphs
print(contacts_graph.degree)

This can be sorted to find the best connected nodes.

In [ ]:
nodes_by_degree = sorted(contacts_graph.degree, key=lambda x: x[1], reverse=True)
print(nodes_by_degree)

The top of the list shows that there are well connected nodes in both the large and the small subunit.

Networkx has a degree histogram function.
The cell below shows that for a bin of 1 (it can be good to increase the bin size for larger graphs).

In [ ]:
bin=1
degree_freq = nx.degree_histogram(contacts_graph)
degrees = range(len(degree_freq))
plt.figure(figsize=(12, 8))
plt.plot(degrees[bin:], degree_freq[bin:],'go-')
plt.xlabel('Degree')
plt.ylabel('Frequency')

The neighbours of particular nodes can be obtained. Here is one of the best connected nodes from the sorted degree dictionary earlier.

In [ ]:
list(contacts_graph.neighbors('eS21'))

Now write code to work to find the node with the highest degree that has any contacts with the other subunit.

In [ ]:
# code to find the nodes with the highest degree that have any contacts with the other subunit.

## Advanced exercise colour  network by degree

This is a slightly tricky, see https://stackoverflow.com/a/28916094 for hint

## Cliques

Cliques are also called 'complete' subgraphs. These are complete groups of nodes where each one is joined to each of the others.   

You can see that some of the best connected nodes are also members of multiple larger cliques.

In [ ]:
list(nx.find_cliques(contacts_graph))

It is interesting to compare a network with an artificial one made in some random way. The original of this is the Erdox-Renyi graph. This is made for the given number of nodes and the probability of edge formation. Conveniently this is the graph density.
Our network had 68 nodes and an approximate density of 0.055.

Of course this will give a different graph every time it is run.

In [ ]:
E = nx.erdos_renyi_graph(68,0.055)

In [ ]:
nx.draw_spring(E,k=0.80,iterations=50, with_labels=True, node_color='r',)

You may see that a low density has a fairly high chance of having disconnected components.
The ribosome did in fact have several disconnect components that were present in the paper's data table but which was editted out for simplicity in the plots for this notebook.

We can plot the degree histogram of the random graph for comparison with the ribosome protein one.

In [ ]:
bin=1
degree_freq = nx.degree_histogram(E)
degrees = range(len(degree_freq))
plt.figure(figsize=(12, 8))
plt.plot(degrees[bin:], degree_freq[bin:],'go-')
plt.xlabel('Degree')
plt.ylabel('Frequency')

The Watts-Strogatz graph is a special kind of random artificial graph. This is made for the given number of nodes and a starting number of edges (k). But then the connections are re-wired to attempt to add a 'small world' connectivity to the network. The parameter p controls the rewiring.

So here is a small world network the same size as our ribosomal one. This version enforces a connected graph.

In [ ]:
S = nx.connected_watts_strogatz_graph(68, k=4, p=0.5)

In [ ]:
nx.draw_spring(S,k=0.80,iterations=50, with_labels=True, node_color='g',)

In [ ]:
bin=1
degree_freq = nx.degree_histogram(S)
degrees = range(len(degree_freq))
plt.figure(figsize=(12, 8))
plt.plot(degrees[bin:], degree_freq[bin:],'go-')
plt.xlabel('Degree')
plt.ylabel('Frequency')

## Advanced exercises: plot network diagram with circle where large and small subunits are in separated sections

This is quite involved.

Even more advanced use a separate circle for the small and large subunits.